# 1. Load the Model from MLflow

In [4]:
import pandas as pd
import mlflow
import mlflow.spark

from pyspark.sql import SparkSession


In [9]:
mlflow.set_tracking_uri("http://localhost:5000")  # Docker MLflow

In [ ]:
# First, verify MLflow tracking server

# print(mlflow.get_tracking_uri())  # Should be "http://localhost:5000"
# client = mlflow.tracking.MlflowClient()
# run = client.get_run("1f44a99a97c942259434b91fbbbfaa47")
# print(run)  # If None → tracking server issue


http://localhost:5000
<Run: data=<RunData: metrics={'f1_df': 1.0}, params={'MulticlassClassificationEvaluator.beta': '1.0',
 'MulticlassClassificationEvaluator.eps': '1e-15',
 'MulticlassClassificationEvaluator.labelCol': 'label_indexed',
 'MulticlassClassificationEvaluator.metricLabel': '0.0',
 'MulticlassClassificationEvaluator.metricName': 'f1',
 'MulticlassClassificationEvaluator.predictionCol': 'prediction',
 'MulticlassClassificationEvaluator.probabilityCol': 'probability',
 'best_DecisionTreeClassifier.maxDepth': '2',
 'collectSubModels': 'False',
 'estimator': 'Pipeline',
 'evaluator': 'MulticlassClassificationEvaluator',
 'parallelism': '1',
 'seed': '-2886568149864525650',
 'trainRatio': '0.75'}, tags={'estimator_class': 'pyspark.ml.tuning.TrainValidationSplit',
 'estimator_name': 'TrainValidationSplit',
 'mlflow.log-model.history': '[{"run_id": "1f44a99a97c942259434b91fbbbfaa47", '
                             '"artifact_path": "model", "utc_time_created": '
                

In [12]:
spark = SparkSession.builder.appName("FruitClassification").getOrCreate()

In [13]:
model_uri = "runs:/1f44a99a97c942259434b91fbbbfaa47/best_fruit_robot"
loaded_model = mlflow.spark.load_model(model_uri)

2026/03/02 06:38:33 INFO mlflow.spark: URI 'runs:/1f44a99a97c942259434b91fbbbfaa47/best_fruit_robot/sparkml' does not point to the current DFS.
2026/03/02 06:38:33 INFO mlflow.spark: File 'runs:/1f44a99a97c942259434b91fbbbfaa47/best_fruit_robot/sparkml' not found on DFS. Will attempt to upload the file.


# 2. Create "Mystery" Data

In [14]:
mystery_data = pd.DataFrame({
    "weight": [165, 52],
    "color": ["Red", "Yellow"]
})
mystery_df = spark.createDataFrame(mystery_data)

# 3. Make Predictions

In [15]:
predictions = loaded_model.transform(mystery_df)
predictions.select("weight", "color", "prediction").show()

+------+------+----------+
|weight| color|prediction|
+------+------+----------+
|   165|   Red|       0.0|
|    52|Yellow|       1.0|
+------+------+----------+

